# 05: The Business Risk Engine (Prediction Intervals)
In this notebook, we simulate our automated deployment environment. 
The Stacker model uses 5 distinct algorithms to predict house prices. Because these models use fundamentally different mathematics (Linear, Geometric, Trees), their disagreement is highly indicative of property risk.
**The Business Logic:**
* 🟢 **GREEN TIER (Auto-List):** Models tightly agree. High confidence.
* 🟡 **YELLOW TIER (Desktop Review):** Moderate disagreement. Flag for human review.
* 🔴 **RED TIER (Human Override):** Severe disagreement (Max/Min Spread > 25%). The algorithm refuses to automate and passes to a Senior Appraiser.


In [1]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')
# 1. Load Test Data
PROC_DIR = "../data/processed/"
FINAL_DIR = "../data/final_for_modeling/"
X_test_strict = pd.read_csv(f"{FINAL_DIR}X_test_strict.csv").fillna(0) # For SVR
X_test_scaled = pd.read_csv(f"{PROC_DIR}X_test_scaled.csv").fillna(0)  # For Ridge
X_test_full = pd.read_csv(f"{FINAL_DIR}X_test_full.csv").fillna(0)     # For RF & LightGBM
X_test_cb = pd.read_csv(f"{PROC_DIR}X_test_clean_text.csv")
cat_cols = X_test_cb.select_dtypes(include=['object']).columns
num_cols = X_test_cb.select_dtypes(exclude=['object']).columns
X_test_cb[cat_cols] = X_test_cb[cat_cols].fillna('Missing')
X_test_cb[num_cols] = X_test_cb[num_cols].fillna(0)  # For CatBoost
y_test = pd.read_csv(f"{PROC_DIR}y_test.csv").squeeze()      # Raw Dollar Targets
# 2. Load 5 Champion Models
ridge = joblib.load("../models/ridge_tuned.pkl")
svr = joblib.load("../models/svr_68_features_tuned.pkl")
rf = joblib.load("../models/random_forest_tuned.pkl")
lgbm = joblib.load("../models/lightgbm_tuned.pkl")
cb = joblib.load("../models/catboost_tuned.pkl")
print("All Data and Models successfully loaded.")


All Data and Models successfully loaded.


In [2]:
def calculate_risk_tier(predictions_dict, weights_dict):
    """
    Calculates final stacked price, epistemic uncertainty (CV), and Risk Tier.
    """
    preds = np.array(list(predictions_dict.values()))
    weights = np.array([weights_dict[k] for k in predictions_dict.keys()])
    
    # Final Stacked Price (Weighted Mean)
    final_price = np.average(preds, weights=weights)
    
    # Weighted Standard Deviation (Epistemic Uncertainty)
    variance = np.average((preds - final_price)**2, weights=weights)
    weighted_std = np.sqrt(variance)
    
    # Risk Metrics
    coef_of_variation = weighted_std / final_price  # % of average disagreement
    max_min_spread = (np.max(preds) - np.min(preds)) / final_price # The 'Rogue Veto'
    
    # Business Logic Thresholds
    if max_min_spread > 0.25 or coef_of_variation > 0.12:
        tier = "🔴 RED (Override)"
    elif max_min_spread > 0.15 or coef_of_variation > 0.07:
        tier = "🟡 YELLOW (Review)"
    else:
        tier = "🟢 GREEN (Auto)"
        
    return final_price, coef_of_variation, max_min_spread, tier


In [3]:
# Approximate optimal V1.0 voting weights (Raw Dollars)
weights = {'Ridge': 0.10, 'SVR': 0.17, 'RF': 0.08, 'LightGBM': 0.45, 'CatBoost': 0.20}
results = []

print("Processing all 292 Test Houses through the Risk Engine...")
for i in range(len(y_test)):
    # Extract row as 2D DataFrame for Scikit-Learn
    house_preds = {
        'Ridge': ridge.predict(X_test_scaled.iloc[[i]])[0],
        'SVR': svr.predict(X_test_strict.iloc[[i]])[0],
        'RF': rf.predict(X_test_full.iloc[[i]])[0],
        'LightGBM': lgbm.predict(X_test_full.iloc[[i]])[0],
        'CatBoost': cb.predict(X_test_cb.iloc[[i]])[0]
    }
    
    final_price, cv, spread, tier = calculate_risk_tier(house_preds, weights)
    
    results.append({
        'Actual Price': y_test.iloc[i],
        'Stacked Price': final_price,
        'Risk Tier': tier,
        'CV (Variance %)': cv,
        'Spread (Max-Min %)': spread,
        **house_preds 
    })

results_df = pd.DataFrame(results)
print("Processing complete!")

Processing all 292 Test Houses through the Risk Engine...
Processing complete!


In [4]:
print("=== FINAL BUSINESS IMPACT REPORT ===")
print("Total Houses Processed: 292\n")

tier_counts = results_df['Risk Tier'].value_counts()
for tier, count in tier_counts.items():
    percentage = (count / 292) * 100
    print(f"{tier}: {count} houses ({percentage:.1f}%)")

print("\n---------------------------------------------------------")
print("EXAMPLE 1: A PERFECT GREEN HOUSE (High Confidence)")
green_house = results_df[results_df['Risk Tier'] == "🟢 GREEN (Auto)"].iloc[0]
print(f"Actual: ${green_house['Actual Price']:,.0f} | Stacked Prediction: ${green_house['Stacked Price']:,.0f}")
print(f"Base Models -> LGBM: ${green_house['LightGBM']:,.0f}, SVR: ${green_house['SVR']:,.0f}, Ridge: ${green_house['Ridge']:,.0f}")

print("\n---------------------------------------------------------")
print("EXAMPLE 2: A RED FLAG ANOMALY (Rogue Veto Triggered)")
red_house = results_df[results_df['Risk Tier'] == "🔴 RED (Override)"].iloc[0]
print(f"Actual: ${red_house['Actual Price']:,.0f} | Stacked Prediction: ${red_house['Stacked Price']:,.0f}")
print(f"Base Models -> LGBM: ${red_house['LightGBM']:,.0f}, SVR: ${red_house['SVR']:,.0f}, Ridge: ${red_house['Ridge']:,.0f}")
print(f"Variance: {red_house['CV (Variance %)']*100:.1f}% | Spread: {red_house['Spread (Max-Min %)']*100:.1f}%")
print("Decision: Algorithms violently disagree. Automation blocked. Sent to Appraiser.")

=== FINAL BUSINESS IMPACT REPORT ===
Total Houses Processed: 292

🟢 GREEN (Auto): 177 houses (60.6%)
🟡 YELLOW (Review): 70 houses (24.0%)
🔴 RED (Override): 45 houses (15.4%)

---------------------------------------------------------
EXAMPLE 1: A PERFECT GREEN HOUSE (High Confidence)
Actual: $154,500 | Stacked Prediction: $141,977
Base Models -> LGBM: $139,893, SVR: $145,160, Ridge: $144,865

---------------------------------------------------------
EXAMPLE 2: A RED FLAG ANOMALY (Rogue Veto Triggered)
Actual: $200,500 | Stacked Prediction: $239,713
Base Models -> LGBM: $250,670, SVR: $191,119, Ridge: $256,726
Variance: 9.5% | Spread: 27.4%
Decision: Algorithms violently disagree. Automation blocked. Sent to Appraiser.
